## Category important words & similarity search

In [1]:
import pandas as pd
import numpy as np
import re
import time
import nltk
#from nltk import bigrams, trigrams
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity


#model = SentenceTransformer('/Users/zphilipp/git/research/relevance/models/sentence-transformer.model')
model = SentenceTransformer('all-MiniLM-L6-v2')

#pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 200)

prepositions_and_conjunctions = [
    "about", "above", "across", "after", "against", "along", "among", "around", "at",
    "before", "behind", "below", "beneath", "beside", "between", "beyond", "by",
    "during", "for", "from", "in", "inside", "into", "near", "of", "off", "on",
    "out", "outside", "over", "through", "throughout", "to", "toward", "under",
    "until", "up", "with", "within", "without", "and", "but", "or", "for", "nor",
    "so", "yet", "although", "because", "as", "since", "unless", "while", "when",
    "where", "after", "before", "the", "a"
]
pattern = r'\b(?:' + '|'.join(prepositions_and_conjunctions) + r')\b'

def remove_prepositions_and_conjunctions(text):
    text = text.lower()
    cleaned_text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r'\d+', '', cleaned_text)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    return cleaned_text.replace("-", "")

/Users/zphilipp/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/zphilipp/miniconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [16]:
df = pd.read_csv('category_data.csv')
df.head()

,attributes.v2_category_name,attributes.v1_category_name,parent,guid,description,name,header,category
0,Shopping,Retail,0ed8f46e-2990-448c-9a8c-50665498a84c,7552494f-2a02-4bf8-91b3-ba34d90debdf,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Personal Care,Shopping
1,NaN,NaN,NaN,2b2dbf9e-86d0-4c7e-915b-408c4564df53,Subscriptions,Subscriptions,NaN,NaN
2,Shopping,Retail,71912b5b-4809-4536-bc03-8ee270060b96,36a0ea5b-210a-452f-82ad-e9b6c96c67ab,Home & Garden - Decorative Vases,Home & Garden - Decorative Vases,Home Decor,Shopping
3,Shopping,Retail,8efb9d2c-0947-44d9-af8f-eec79a5a3fc2,2073934d-da10-4cdf-bed1-59bb12e9d00a,Apparel & Accessories - Girls - Belts,Apparel & Accessories - Girls - Belts,Clothing Accessories,Shopping
4,Shopping,Retail,d05bb230-5f44-4180-9f9b-323f72649552,508d94c4-3851-4b51-b0c0-4efcf6381fcb,Action Figure / Doll,Action Figure / Doll,Toys & Hobbies,Shopping


In [17]:
df['text'] = df['description'] + '. ' + df['name']
df['text'] = df['text'].apply(remove_prepositions_and_conjunctions)
df_ = df
df_.head()

,attributes.v2_category_name,attributes.v1_category_name,parent,guid,description,name,header,category,text
0,Shopping,Retail,0ed8f46e-2990-448c-9a8c-50665498a84c,7552494f-2a02-4bf8-91b3-ba34d90debdf,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Personal Care,Shopping,health & beauty deodorant & antiperspirant mens deodorant. health & beauty deodorant & antiperspirant mens deodorant
1,NaN,NaN,NaN,2b2dbf9e-86d0-4c7e-915b-408c4564df53,Subscriptions,Subscriptions,NaN,NaN,subscriptions. subscriptions
2,Shopping,Retail,71912b5b-4809-4536-bc03-8ee270060b96,36a0ea5b-210a-452f-82ad-e9b6c96c67ab,Home & Garden - Decorative Vases,Home & Garden - Decorative Vases,Home Decor,Shopping,home & garden decorative vases. home & garden decorative vases
3,Shopping,Retail,8efb9d2c-0947-44d9-af8f-eec79a5a3fc2,2073934d-da10-4cdf-bed1-59bb12e9d00a,Apparel & Accessories - Girls - Belts,Apparel & Accessories - Girls - Belts,Clothing Accessories,Shopping,apparel & accessories girls belts. apparel & accessories girls belts
4,Shopping,Retail,d05bb230-5f44-4180-9f9b-323f72649552,508d94c4-3851-4b51-b0c0-4efcf6381fcb,Action Figure / Doll,Action Figure / Doll,Toys & Hobbies,Shopping,action figure / doll. action figure / doll


#### Get all titles from Deals and Options text

#### Create word embedings and transform data

In [18]:
#df_['text'] = df_['text'].tolist()
df_['embeddings'] = df_['text'].apply(lambda x: model.encode(x))

combined_embeddings = np.array(df_['embeddings'].tolist())

In [13]:
df_['text']

0        health & beauty  deodorant & antiperspirant  mens  deodorant. health & beauty  deodorant & antiperspirant  mens  deodorant
1                                                                                                      subscriptions. subscriptions
2                                                                  home & garden  decorative vases. home & garden  decorative vases
3                                                          apparel & accessories  girls  belts. apparel & accessories  girls  belts
4                                                                                        action figure / doll. action figure / doll
                                                                    ...                                                            
10377                                                      apparel & accessories  mens  gloves. apparel & accessories  mens  gloves
10378                                                                       

In [19]:
df_.head()
#df_.count()

,attributes.v2_category_name,attributes.v1_category_name,parent,guid,description,name,header,category,text,embeddings
0,Shopping,Retail,0ed8f46e-2990-448c-9a8c-50665498a84c,7552494f-2a02-4bf8-91b3-ba34d90debdf,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Personal Care,Shopping,health & beauty deodorant & antiperspirant mens deodorant. health & beauty deodorant & antiperspirant mens deodorant,"[-0.0009589831, 0.0081701465, 0.10928768, 0.015979808, 0.079705745, -0.054364037, 0.08232258, -0.03318259, -0.06707983, -0.013876773, 0.06261128, -0.051048968, -0.01748341, -0.05335682, 0.11192895..."
1,NaN,NaN,NaN,2b2dbf9e-86d0-4c7e-915b-408c4564df53,Subscriptions,Subscriptions,NaN,NaN,subscriptions. subscriptions,"[-0.051509548, -0.06444815, -0.04112121, 0.042434275, -0.0010615113, 0.035495438, 0.08559268, -0.03848302, 0.07432327, 0.0006714391, 0.021693816, 0.03552054, 0.06964188, 0.014078247, -0.004548607,..."
2,Shopping,Retail,71912b5b-4809-4536-bc03-8ee270060b96,36a0ea5b-210a-452f-82ad-e9b6c96c67ab,Home & Garden - Decorative Vases,Home & Garden - Decorative Vases,Home Decor,Shopping,home & garden decorative vases. home & garden decorative vases,"[0.051288802, 0.020537999, 0.06493606, -0.058544688, -0.058173075, 0.027262967, 0.05664413, -0.015885562, -0.040061068, 0.026442869, -0.040510807, -0.0399215, -0.019836022, 0.03982401, 0.06903698,..."
3,Shopping,Retail,8efb9d2c-0947-44d9-af8f-eec79a5a3fc2,2073934d-da10-4cdf-bed1-59bb12e9d00a,Apparel & Accessories - Girls - Belts,Apparel & Accessories - Girls - Belts,Clothing Accessories,Shopping,apparel & accessories girls belts. apparel & accessories girls belts,"[0.011771777, -0.00048552422, 0.0127890995, 0.012789315, -0.027351007, -0.04309254, 0.08880087, -0.010798826, -0.043110605, -0.0079502845, 0.115006246, -0.009183229, 0.10886563, -0.042723708, 0.05..."
4,Shopping,Retail,d05bb230-5f44-4180-9f9b-323f72649552,508d94c4-3851-4b51-b0c0-4efcf6381fcb,Action Figure / Doll,Action Figure / Doll,Toys & Hobbies,Shopping,action figure / doll. action figure / doll,"[-0.013929551, -0.06954958, 0.0118156215, -0.0005743038, -0.02140315, 0.0012134686, 0.06832989, 0.009302696, 0.03917064, 0.058662686, 0.09623468, -0.003958322, 0.0002645173, 0.06230961, 0.06897391..."


In [11]:
def query_embedding_reduce(query_embedding):
    if query_embedding.shape[1] > 384:
        query_embedding_reduced = np.mean(query_embedding.reshape(-1, 2, 384), axis=1)
    else:
        query_embedding_reduced = query_embedding
    return query_embedding_reduced

df_[['guid', 'name', 'category', 'text', 'embeddings']].to_csv('models/category_embeding.csv')
df_.head(5)

,attributes.v2_category_name,attributes.v1_category_name,parent,guid,description,name,header,category,text,embeddings
0,Shopping,Retail,0ed8f46e-2990-448c-9a8c-50665498a84c,7552494f-2a02-4bf8-91b3-ba34d90debdf,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Personal Care,Shopping,health & beauty deodorant & antiperspirant mens deodorant. health & beauty deodorant & antiperspirant mens deodorant,"[-0.0009589831, 0.0081701465, 0.10928768, 0.015979808, 0.079705745, -0.054364037, 0.08232258, -0.03318259, -0.06707983, -0.013876773, 0.06261128, -0.051048968, -0.01748341, -0.05335682, 0.11192895..."
1,NaN,NaN,NaN,2b2dbf9e-86d0-4c7e-915b-408c4564df53,Subscriptions,Subscriptions,NaN,NaN,subscriptions. subscriptions,"[-0.051509548, -0.06444815, -0.04112121, 0.042434275, -0.0010615113, 0.035495438, 0.08559268, -0.03848302, 0.07432327, 0.0006714391, 0.021693816, 0.03552054, 0.06964188, 0.014078247, -0.004548607,..."
2,Shopping,Retail,71912b5b-4809-4536-bc03-8ee270060b96,36a0ea5b-210a-452f-82ad-e9b6c96c67ab,Home & Garden - Decorative Vases,Home & Garden - Decorative Vases,Home Decor,Shopping,home & garden decorative vases. home & garden decorative vases,"[0.051288802, 0.020537999, 0.06493606, -0.058544688, -0.058173075, 0.027262967, 0.05664413, -0.015885562, -0.040061068, 0.026442869, -0.040510807, -0.0399215, -0.019836022, 0.03982401, 0.06903698,..."
3,Shopping,Retail,8efb9d2c-0947-44d9-af8f-eec79a5a3fc2,2073934d-da10-4cdf-bed1-59bb12e9d00a,Apparel & Accessories - Girls - Belts,Apparel & Accessories - Girls - Belts,Clothing Accessories,Shopping,apparel & accessories girls belts. apparel & accessories girls belts,"[0.011771777, -0.00048552422, 0.0127890995, 0.012789315, -0.027351007, -0.04309254, 0.08880087, -0.010798826, -0.043110605, -0.0079502845, 0.115006246, -0.009183229, 0.10886563, -0.042723708, 0.05..."
4,Shopping,Retail,d05bb230-5f44-4180-9f9b-323f72649552,508d94c4-3851-4b51-b0c0-4efcf6381fcb,Action Figure / Doll,Action Figure / Doll,Toys & Hobbies,Shopping,action figure / doll. action figure / doll,"[-0.013929551, -0.06954958, 0.0118156215, -0.0005743038, -0.02140315, 0.0012134686, 0.06832989, 0.009302696, 0.03917064, 0.058662686, 0.09623468, -0.003958322, 0.0002645173, 0.06230961, 0.06897391..."


In [20]:
def get_top_similarity(query_embedding_reduced, combined_embeddings):
    similarities = cosine_similarity(query_embedding_reduced, combined_embeddings).flatten()
    closest_indices = np.argsort(similarities)[-10:]

    closest_rows = []
    for index in reversed(closest_indices):
        if similarities[index] > 0.3:
        
            closest_rows.append([df_.iloc[index], similarities[index]])

    return closest_rows

### Test query -> category use Cosine similarity of category embedings and query embedings

In [21]:
def get_sim(query):
    start_time = time.time()
    query_embedding_reduced = query_embedding_reduce(model.encode(query).reshape(1, -1))
    print (f"Embeding time :{time.time() - start_time}")
    result = get_top_similarity(query_embedding_reduced, combined_embeddings)
    print (f"Total run time :{time.time() - start_time}")
    for row in result:
        print(f"Closest Category: <{row[0]['name']}> -> score {row[1]}")

In [22]:
get_sim(['massage', 'oil'])

Embeding time :0.25776076316833496
Total run time :0.31685304641723633
Closest Category: <Massage> -> score 0.7837384939193726
Closest Category: <Massage> -> score 0.7837384939193726
Closest Category: <Massage - Aroma Oil> -> score 0.7545453310012817
Closest Category: <Massage - Hydro> -> score 0.6603992581367493
Closest Category: <Massage - Relaxation> -> score 0.6500920057296753
Closest Category: <Massage - Full Body> -> score 0.6429497599601746
Closest Category: <Massage - Honey> -> score 0.6275819540023804
Closest Category: <Massage Furniture> -> score 0.6233825087547302
Closest Category: <Massage - Chocolate> -> score 0.6206730008125305
Closest Category: <Massage & Relaxation Products> -> score 0.6186055541038513


In [23]:
get_sim(['oil'])

Embeding time :0.16373395919799805
Total run time :0.16800308227539062
Closest Category: <Oil Change> -> score 0.6687585115432739
Closest Category: <Massage - Aroma Oil> -> score 0.5527456402778625
Closest Category: <CBD Oil> -> score 0.4987090528011322
Closest Category: <Oil Change - Full Service> -> score 0.4907195270061493
Closest Category: <Condiment / Vinegar / Oil> -> score 0.48658865690231323
Closest Category: <Water> -> score 0.47271132469177246
Closest Category: <Engine> -> score 0.45922207832336426
Closest Category: <Shoes> -> score 0.4355558454990387
Closest Category: <Fries> -> score 0.4338662624359131
Closest Category: <Marijuana> -> score 0.4209069311618805


In [24]:
get_sim(['change'])

Embeding time :0.11089324951171875
Total run time :0.11732721328735352
Closest Category: <Oil Change> -> score 0.5119956135749817
Closest Category: <Moving> -> score 0.4426957368850708
Closest Category: <eLearning - Change Management> -> score 0.4392528831958771
Closest Category: <Tire Change / Replacement> -> score 0.3982493281364441
Closest Category: <Tire / Tyre Change / Replacement> -> score 0.3855496048927307
Closest Category: <Alternative> -> score 0.37698429822921753
Closest Category: <Knife> -> score 0.36868393421173096
Closest Category: <Water> -> score 0.3469914197921753
Closest Category: <Live Food> -> score 0.3422635793685913
Closest Category: <Alterations & Cobbler> -> score 0.33561843633651733


In [25]:
get_sim(['oil', 'change'])

Embeding time :0.06586694717407227
Total run time :0.07306098937988281
Closest Category: <Oil Change> -> score 0.7310999631881714
Closest Category: <Water> -> score 0.5075439214706421
Closest Category: <Oil Change - Full Service> -> score 0.48798999190330505
Closest Category: <Engine> -> score 0.461206316947937
Closest Category: <Shoes> -> score 0.45931997895240784
Closest Category: <Moving> -> score 0.4489561915397644
Closest Category: <Fries> -> score 0.4375988841056824
Closest Category: <Painting> -> score 0.4313236474990845
Closest Category: <Knife> -> score 0.4309301972389221
Closest Category: <Makeup> -> score 0.4261842966079712


In [26]:
get_sim(['sauna', 'massage'])

Embeding time :0.18359684944152832
Total run time :0.19285368919372559
Closest Category: <Sauna> -> score 0.813665509223938
Closest Category: <Massage> -> score 0.7961447834968567
Closest Category: <Massage> -> score 0.7961447834968567
Closest Category: <Massage - Relaxation> -> score 0.7327099442481995
Closest Category: <Massage - Full Body> -> score 0.7085054516792297
Closest Category: <Massage - Chair> -> score 0.6867082715034485
Closest Category: <Massage Furniture> -> score 0.6817094087600708
Closest Category: <Massage Course> -> score 0.6711677312850952
Closest Category: <Massage - Sports> -> score 0.6555063724517822
Closest Category: <Massage - Deep Tissue> -> score 0.6531846523284912


In [27]:
get_sim(['oil', 'massage'])

Embeding time :0.09050107002258301
Total run time :0.1030879020690918
Closest Category: <Massage> -> score 0.7837384939193726
Closest Category: <Massage> -> score 0.7837384939193726
Closest Category: <Massage - Aroma Oil> -> score 0.7545453310012817
Closest Category: <Massage - Hydro> -> score 0.6603992581367493
Closest Category: <Massage - Relaxation> -> score 0.6500920057296753
Closest Category: <Massage - Full Body> -> score 0.6429497599601746
Closest Category: <Massage - Honey> -> score 0.6275819540023804
Closest Category: <Massage Furniture> -> score 0.6233825087547302
Closest Category: <Massage - Chocolate> -> score 0.6206730008125305
Closest Category: <Massage & Relaxation Products> -> score 0.6186055541038513


In [28]:
get_sim(['massage', 'oil'])

Embeding time :0.12324094772338867
Total run time :0.15114808082580566
Closest Category: <Massage> -> score 0.7837384939193726
Closest Category: <Massage> -> score 0.7837384939193726
Closest Category: <Massage - Aroma Oil> -> score 0.7545453310012817
Closest Category: <Massage - Hydro> -> score 0.6603992581367493
Closest Category: <Massage - Relaxation> -> score 0.6500920057296753
Closest Category: <Massage - Full Body> -> score 0.6429497599601746
Closest Category: <Massage - Honey> -> score 0.6275819540023804
Closest Category: <Massage Furniture> -> score 0.6233825087547302
Closest Category: <Massage - Chocolate> -> score 0.6206730008125305
Closest Category: <Massage & Relaxation Products> -> score 0.6186055541038513


In [29]:
get_sim(['valvoline', 'oil'])

Embeding time :0.1262950897216797
Total run time :0.1335279941558838
Closest Category: <Oil Change> -> score 0.4918593168258667
Closest Category: <Massage - Aroma Oil> -> score 0.4836856424808502
Closest Category: <Engine> -> score 0.42214420437812805
Closest Category: <Pizza> -> score 0.40887051820755005
Closest Category: <Shoes> -> score 0.4001413881778717
Closest Category: <Tank> -> score 0.39355432987213135
Closest Category: <Grill> -> score 0.3909233808517456
Closest Category: <Oil Change - Full Service> -> score 0.38998734951019287
Closest Category: <Meat> -> score 0.38863083720207214
Closest Category: <Wine> -> score 0.38466084003448486


In [30]:
get_sim(['water'])

Embeding time :0.10050010681152344
Total run time :0.10704421997070312
Closest Category: <Water> -> score 0.9213337302207947
Closest Category: <Water Cooler> -> score 0.6574827432632446
Closest Category: <Skiing - Water> -> score 0.6412049531936646
Closest Category: <Drinks> -> score 0.5966424942016602
Closest Category: <Juice> -> score 0.5908203721046448
Closest Category: <Swimming> -> score 0.5804554224014282
Closest Category: <Swimming> -> score 0.5804554224014282
Closest Category: <Beverages> -> score 0.5734614133834839
Closest Category: <Beverage> -> score 0.5720568895339966
Closest Category: <Beer> -> score 0.5521512627601624


In [31]:
get_sim(['water', 'parks'])

Embeding time :0.0655829906463623
Total run time :0.07216596603393555
Closest Category: <Parks> -> score 0.7367718815803528
Closest Category: <Water> -> score 0.7044963836669922
Closest Category: <Waterpark> -> score 0.6598027944564819
Closest Category: <Pool> -> score 0.601981520652771
Closest Category: <Waterpark Resort> -> score 0.590696394443512
Closest Category: <Amusement Park> -> score 0.5613821744918823
Closest Category: <Hotel - Waterparks> -> score 0.5521727800369263
Closest Category: <Swimming> -> score 0.5507572889328003
Closest Category: <Swimming> -> score 0.5507572889328003
Closest Category: <Waterpark Resort - Beach> -> score 0.5483187437057495


In [32]:
get_sim(['amc'])

Embeding time :0.0658118724822998
Total run time :0.07475900650024414
Closest Category: <Auditorium> -> score 0.4826323390007019
Closest Category: <Cinema> -> score 0.470320463180542
Closest Category: <Hotel - Hobart - Cruises> -> score 0.46727436780929565
Closest Category: <Cinema / Movie Theater> -> score 0.45210981369018555
Closest Category: <Hotel - Hobart - Casino Resorts> -> score 0.45180243253707886
Closest Category: <Circus> -> score 0.44649845361709595
Closest Category: <Hotel - Hobart - Parks> -> score 0.4420951008796692
Closest Category: <Hotel - Hobart - Camping> -> score 0.44140198826789856
Closest Category: <Tour - Hobart - Cruises> -> score 0.4348657429218292
Closest Category: <Cinema - Open Air> -> score 0.4277368485927582


In [33]:
get_sim(['pilates'])

Embeding time :0.12116384506225586
Total run time :0.13131070137023926
Closest Category: <Pilates - Mat> -> score 0.7964856624603271
Closest Category: <Pilates - Equipment> -> score 0.7522004842758179
Closest Category: <Sporting Goods - Yoga & Pilates> -> score 0.6775169968605042
Closest Category: <Sporting Goods - Yoga & Pilates - Reformers> -> score 0.6402606964111328
Closest Category: <Sporting Goods - Yoga & Pilates - Flexbands> -> score 0.6250981092453003
Closest Category: <Sporting Goods - Yoga & Pilates - Blocks> -> score 0.6043626666069031
Closest Category: <Sporting Goods - Yoga & Pilates - Chairs> -> score 0.6014114022254944
Closest Category: <Apparel & Accessories - Activewear - Yoga & Pilates> -> score 0.5767750144004822
Closest Category: <Sporting Goods - Yoga & Pilates - Mats> -> score 0.5767335295677185
Closest Category: <Sporting Goods - Yoga & Pilates - Foam Wedges> -> score 0.5695319175720215


In [34]:
get_sim(['ring'])

Embeding time :0.0771951675415039
Total run time :0.09314203262329102
Closest Category: <Custom - Rings> -> score 0.6219835877418518
Closest Category: <Sporting Goods - Gymnastics Rings> -> score 0.5747995972633362
Closest Category: <Jewelry> -> score 0.5688859820365906
Closest Category: <Ideeli - Accessories - Fine Metal Jewelry - Rings> -> score 0.5620177388191223
Closest Category: <Apparel & Accessories - Rings> -> score 0.5509170293807983
Closest Category: <Apparel & Accessories - Jewelry - Rings - Womens> -> score 0.5388330221176147
Closest Category: <Apparel & Accessories - Jewelry - Rings - Mens> -> score 0.533768892288208
Closest Category: <Apparel & Accessories - Jewelry - Rings - Girls> -> score 0.5249111652374268
Closest Category: <Apparel & Accessories - Jewelry - Rings - Boys> -> score 0.518591046333313
Closest Category: <Ideeli - Accessories - Diamond Jewelry - Rings> -> score 0.5176674127578735


In [35]:
get_sim(['wheel'])

Embeding time :0.10980510711669922
Total run time :0.118682861328125
Closest Category: <Wheels & Tires> -> score 0.6518899202346802
Closest Category: <Wheel Restoration> -> score 0.6274017095565796
Closest Category: <Sporting Goods - Cycling - Wheels> -> score 0.6239476203918457
Closest Category: <Tires> -> score 0.6190994381904602
Closest Category: <Ferris Wheel / Panoramic Wheel> -> score 0.5857815742492676
Closest Category: <Spinning> -> score 0.5753920078277588
Closest Category: <Automotive - Tires and Wheels> -> score 0.559571385383606
Closest Category: <Sporting Goods - Skateboarding - Wheels> -> score 0.5580050945281982
Closest Category: <Bicycle> -> score 0.549507737159729
Closest Category: <Automotive - Replacement Parts - Wheels> -> score 0.5453565120697021


In [36]:
get_sim(['nail'])

Embeding time :0.05463600158691406
Total run time :0.06144094467163086
Closest Category: <Nail Care> -> score 0.7289608716964722
Closest Category: <Nail Services> -> score 0.6860638856887817
Closest Category: <Nail Design> -> score 0.6837736368179321
Closest Category: <Manicure> -> score 0.6089181303977966
Closest Category: <BB - Manicure> -> score 0.5800713300704956
Closest Category: <Health & Beauty - Nail Care> -> score 0.5341607332229614
Closest Category: <Home Improvement - Nails & Screws - Nails> -> score 0.5302634835243225
Closest Category: <Home Improvement - Nails & Screws> -> score 0.47619688510894775
Closest Category: <Steak> -> score 0.4738704562187195
Closest Category: <BB - Manicure - Shellac / No-Chip / Gel> -> score 0.4671249985694885


In [37]:
get_sim(['massage', 'palace'])

Embeding time :0.06431317329406738
Total run time :0.07361006736755371
Closest Category: <Massage> -> score 0.7401810884475708
Closest Category: <Massage> -> score 0.7401810884475708
Closest Category: <Massage - Chair> -> score 0.6681030988693237
Closest Category: <Massage Course> -> score 0.6548519134521484
Closest Category: <Massage Furniture> -> score 0.6546955704689026
Closest Category: <Massage - Oriental> -> score 0.6511597633361816
Closest Category: <Massage - Relaxation> -> score 0.6466529369354248
Closest Category: <Massage - Sports> -> score 0.6449441313743591
Closest Category: <Massage - Full Body> -> score 0.6390796899795532
Closest Category: <Massage - African> -> score 0.6119297742843628


In [38]:
get_sim(['apple', 'cider'])

Embeding time :0.09135317802429199
Total run time :0.10143494606018066
Closest Category: <Apple Picking> -> score 0.6202783584594727
Closest Category: <Fruit> -> score 0.5224423408508301
Closest Category: <Cutlery> -> score 0.4872254729270935
Closest Category: <Juice> -> score 0.4827001094818115
Closest Category: <Computer> -> score 0.4766603112220764
Closest Category: <Electronics - Laptops - Mac> -> score 0.45717114210128784
Closest Category: <Produce> -> score 0.4509314298629761
Closest Category: <Fruit Picking> -> score 0.4352479577064514
Closest Category: <Electronics - Laptops - Business Mac> -> score 0.43194878101348877
Closest Category: <Wine> -> score 0.429563969373703
